# Stage 09 / Lecture 57: LangChain v1+ Ecosystem and UV Project Setup

This production reference notebook verifies and inspects the modernized **LangChain v1+** runtime ecosystem alongside the **UV** Rust-based Python package manager.

### Core Objectives
1. Inspect active environment tooling (`uv` version, Python runtime, `pyproject.toml` configuration).
2. Verify LangChain v1+ core module availability (`init_chat_model`, `create_agent`, `langchain.agents.middleware`).
3. Validate the standardized message schema hierarchy (`SystemMessage`, `HumanMessage`, `AIMessage`, `ToolMessage`).
4. Demonstrate multi-provider model configuration patterns and structured tool inspection.


In [1]:
from typing import Dict, List, Any, Optional
import subprocess
import sys
from pathlib import Path

import langchain
import langchain_core
import pydantic
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

print(f"Python Runtime: {sys.version.split()[0]}")
print(f"LangChain Version: {langchain.__version__}")
print(f"LangChain Core Version: {langchain_core.__version__}")
print(f"Pydantic Version: {pydantic.__version__}")


Python Runtime: 3.12.12
LangChain Version: 1.4.3
LangChain Core Version: 1.6.6
Pydantic Version: 2.13.5


### 1. UV Package Manager & Workspace Integrity Audit

Modern AI production projects utilize **UV** (written in Rust) for deterministic dependency locking (`uv.lock`) and sub-second virtual environment synchronization. Here we programmatically inspect the active `uv` binary and locate `pyproject.toml`.


In [2]:
def verify_uv_environment() -> Dict[str, Any]:
    """Audits local UV installation and locates workspace pyproject.toml."""
    try:
        uv_proc = subprocess.run(
            ["uv", "--version"],
            capture_output=True,
            text=True,
            check=True
        )
        uv_version = uv_proc.stdout.strip()
    except (subprocess.SubprocessError, FileNotFoundError) as exc:
        raise RuntimeError(f"UV package manager is unavailable: {exc}") from exc

    candidate_roots = [Path("."), Path(".."), Path("../.."), Path("../../..")]
    pyproject_path = next((p / "pyproject.toml" for p in candidate_roots if (p / "pyproject.toml").exists()), None)
    lockfile_path = next((p / "uv.lock" for p in candidate_roots if (p / "uv.lock").exists()), None)

    assert pyproject_path is not None, "pyproject.toml could not be located in workspace hierarchy."
    return {
        "uv_version": uv_version,
        "pyproject_found": str(pyproject_path.resolve()),
        "lockfile_present": lockfile_path is not None
    }

env_audit = verify_uv_environment()
assert "uv" in env_audit["uv_version"].lower()
assert env_audit["lockfile_present"] is True

for key, val in env_audit.items():
    print(f"{key}: {val}")


uv_version: uv 0.13.1 (62faed335 2026-10-13 x86_64-pc-windows-msvc)
pyproject_found: D:\Code\AI\rag-mastery\pyproject.toml
lockfile_present: True


### 2. Inspecting LangChain v1+ Core APIs and Middleware Registry

LangChain v1+ consolidates agent execution around unified constructors (`create_agent`), provider-agnostic model factories (`init_chat_model`), and composable execution middlewares (`langchain.agents.middleware`).


In [3]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
import langchain.agents.middleware as lc_middleware

major_version = int(langchain.__version__.split(".")[0])
assert major_version >= 1, f"Expected LangChain v1+, found {langchain.__version__}"

expected_middlewares = [
    "AgentMiddleware",
    "HumanInTheLoopMiddleware",
    "ContextEditingMiddleware",
    "ModelCallLimitMiddleware",
    "LLMToolSelectorMiddleware"
]

available_middlewares = [m for m in expected_middlewares if hasattr(lc_middleware, m)]
assert len(available_middlewares) == len(expected_middlewares), "Missing core LangChain v1 middleware exports."

print("Verified LangChain v1+ Core Constructors:")
print(f"  - init_chat_model: {callable(init_chat_model)}")
print(f"  - create_agent: {callable(create_agent)}")
print(f"Verified Built-in Middlewares ({len(available_middlewares)}):")
for mw in available_middlewares:
    print(f"  - {mw}")


Verified LangChain v1+ Core Constructors:
  - init_chat_model: True
  - create_agent: True
Verified Built-in Middlewares (5):
  - AgentMiddleware
  - HumanInTheLoopMiddleware
  - ContextEditingMiddleware
  - ModelCallLimitMiddleware
  - LLMToolSelectorMiddleware


### 3. Standardized Message Hierarchy Validation

In modern agent runtimes, conversation trajectories and tool executions rely on explicitly typed message objects (`SystemMessage`, `HumanMessage`, `AIMessage`, `ToolMessage`).


In [4]:
def build_sample_agent_trajectory() -> List[Any]:
    """Constructs a typed multi-turn agent message sequence including tool invocation."""
    sys_msg = SystemMessage(content="You are an enterprise RAG assistant.")
    user_msg = HumanMessage(content="Check the quarterly revenue status for Q3.")
    ai_tool_call = AIMessage(
        content="",
        tool_calls=[{
            "name": "query_financial_db",
            "args": {"quarter": "Q3"},
            "id": "call_q3_rev_001"
        }]
    )
    tool_result = ToolMessage(
        content='{"quarter": "Q3", "revenue_usd": 30000000}',
        tool_call_id="call_q3_rev_001"
    )
    ai_final = AIMessage(content="Q3 revenue reached $30,000,000.")
    return [sys_msg, user_msg, ai_tool_call, tool_result, ai_final]

trajectory = build_sample_agent_trajectory()
assert len(trajectory) == 5
assert trajectory[2].tool_calls[0]["name"] == "query_financial_db"
assert trajectory[3].tool_call_id == trajectory[2].tool_calls[0]["id"]

print("Agent Message Trajectory Validated:")
for idx, msg in enumerate(trajectory, 1):
    role = type(msg).__name__
    preview = msg.content if msg.content else f"[Tool Call -> {msg.tool_calls[0]['name']}]"
    print(f"  [{idx}] {role}: {preview}")

print("All Stage 09 / Lecture 57 ecosystem invariants passed.")


Agent Message Trajectory Validated:
  [1] SystemMessage: You are an enterprise RAG assistant.
  [2] HumanMessage: Check the quarterly revenue status for Q3.
  [3] AIMessage: [Tool Call -> query_financial_db]
  [4] ToolMessage: {"quarter": "Q3", "revenue_usd": 30000000}
  [5] AIMessage: Q3 revenue reached $30,000,000.
All Stage 09 / Lecture 57 ecosystem invariants passed.
